In [ ]:
#TOOL CREATION 
from langchain_community.utilities import ArxivAPIWrapper,WikipediaAPIWrapper
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun 

In [ ]:
#built in tool- wiki

api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [5]:
#built in tool- wiki
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [6]:
tools = [arxiv,wiki]

In [10]:
#CUSTOM TOOL CREATION 
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.vectorstores import FAISS

In [8]:
import os 
from dotenv import load_dotenv
load_dotenv() 

os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [11]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
final_docs = text_splitter.split_documents(docs)
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(final_docs,embeddings)
retriever = db.as_retriever()
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1610.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A47B9A80B0>, search_kwargs={})

In [12]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")
retriever_tool

StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001A401E2C360>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001A401E2C400>)

In [13]:
tools.append(retriever_tool)
tools

[ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\HP\\Udemy\\Ai\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001A401E2C360>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001A401E2C400>)]

In [32]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq

groq_api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")

In [ ]:
# from langchainhub.client import Client

# client = Client()
# prompt = client.pull("hwchase17/openai-functions-agent")

# print(prompt)

C:\Users\HP\AppData\Local\Temp\ipykernel_15128\965164910.py:4: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("hwchase17/openai-functions-agent")
c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "SystemMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are a helpful assistant", "input_variables": [], "template_format": "f-string", "partial_variables": {}}}}}, {"id": ["langchain", "prompts", "chat", "MessagesPlaceholder"], "lc": 1, "type": "constructor", "kwargs": {"optional": true, "variable_name": "chat_history"}}, {"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "{input}", "input_variables": ["input"], "template_format": "f-string", "partial_variables": {}}}}}, {"id": ["langchain", "pro

In [ ]:
#prompt =  ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant"),
#     MessagesPlaceholder(variable_name="chat_history", optional=True),
#     ("human", "{input}"),
#     MessagesPlaceholder(variable_name="agent_scratchpad")
# ])

In [33]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant"
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is AI?"}
    ]
})

print(response)

{'messages': [HumanMessage(content='What is AI?', additional_kwargs={}, response_metadata={}, id='4b955b9a-7458-4ad1-a9ba-019489fc58ce'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vy1v663nf', 'function': {'arguments': '{"query":"AI"}', 'name': 'wikipedia'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 502, 'total_tokens': 516, 'completion_time': 0.034921211, 'completion_tokens_details': None, 'prompt_time': 0.046274868, 'prompt_tokens_details': None, 'queue_time': 0.046558023, 'total_time': 0.081196079}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d47b9-fe37-7453-a3f9-88deaca85975-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'AI'}, 'id': 'vy1v663nf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 502, 'output_tokens': 

In [34]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Tell me about langsmith?"}
    ]
})

print(response)

{'messages': [HumanMessage(content='Tell me about langsmith?', additional_kwargs={}, response_metadata={}, id='f0204015-c577-4eda-8e99-a9ae0a22a7f3'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'z0zmje6bw', 'function': {'arguments': '{"query":"langsmith"}', 'name': 'langsmith-search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 504, 'total_tokens': 521, 'completion_time': 0.021740101, 'completion_tokens_details': None, 'prompt_time': 0.041789634, 'prompt_tokens_details': None, 'queue_time': 0.046816675, 'total_time': 0.063529735}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d47c2-faa1-7de3-af5e-845eefc9014e-0', tool_calls=[{'name': 'langsmith-search', 'args': {'query': 'langsmith'}, 'id': 'z0zmje6bw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

In [36]:
#to maintain chat history
# chat_history = []

# # First user message
# chat_history.append({"role": "user", "content": "What is AI?"})

# response = agent.invoke({
#     "messages": chat_history
# })

# # Add assistant response
# chat_history.append({
#     "role": "assistant",
#     "content": response["messages"][-1].content
# })